# FAIR²-FoodHealth Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the **FAIR²-FoodHealth** dataset using the `mlcroissant` library.

The dataset is described by a Croissant schema and contains provenance-rich, machine-readable records related to foods, nutrients, mappings, governance, and food-health guidance.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json`

### Important note
In this notebook, dataset entities such as record sets and fields are referenced using their Croissant `@id` values whenever available. Because this dataset may expose metadata differently depending on the installed `mlcroissant` version, the notebook includes a few defensive inspection utilities to discover available `@id` values before loading data.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
We begin by importing the required libraries, defining the dataset URL in a variable, and loading the Croissant metadata using `mlcroissant`.

Per the notebook requirements, `dataset.metadata` is handled as a single object rather than being subscripted directly.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.3yz2-mb2d/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON-like Python data
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset loaded successfully.\n")
print("Name:", metadata_json.get("name", "N/A"))
print("Identifier:", metadata_json.get("identifier", "N/A"))
print("Version:", metadata_json.get("version", "N/A"))
print("License:", metadata_json.get("license", "N/A"))
print("Date published:", metadata_json.get("datePublished", "N/A"))
print("\nDescription:\n", metadata_json.get("description", "N/A"))

## 2. Metadata Overview
In this section, we inspect the dataset-level metadata and discover available record sets.

The FAIR² package description included with this prompt shows `recordSet: []`, but the Croissant URL may still expose record sets through the parser. To make the notebook robust, we inspect both the JSON metadata and the `mlcroissant` dataset object.

In [ ]:
# High-level metadata summary
summary = {
    "@id": metadata_json.get("@id"),
    "name": metadata_json.get("name"),
    "identifier": metadata_json.get("identifier"),
    "version": metadata_json.get("version"),
    "keywords": metadata_json.get("keywords", []),
    "distribution_count": len(metadata_json.get("distribution", [])),
    "recordSet_count_in_json": len(metadata_json.get("recordSet", [])),
}

pprint(summary)

In [ ]:
# Helper functions to inspect available record sets and field metadata safely
def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def extract_id_name(obj):
    if hasattr(obj, "to_json"):
        obj = obj.to_json()
    if isinstance(obj, dict):
        return {
            "@id": obj.get("@id"),
            "name": obj.get("name"),
            "description": obj.get("description")
        }
    return {"@id": None, "name": str(obj), "description": None}

def get_recordset_candidates(ds, metadata_dict):
    candidates = []

    # From metadata JSON
    for rs in as_list(metadata_dict.get("recordSet")):
        candidates.append(extract_id_name(rs))

    # From dataset object attributes, if present
    for attr in ["record_sets", "recordsets", "records_sets", "metadata"]:
        if hasattr(ds, attr):
            try:
                value = getattr(ds, attr)
                if attr == "metadata":
                    if hasattr(value, "to_json"):
                        value_json = value.to_json()
                        for rs in as_list(value_json.get("recordSet")):
                            candidates.append(extract_id_name(rs))
                else:
                    for rs in as_list(value):
                        candidates.append(extract_id_name(rs))
            except Exception:
                pass

    # Deduplicate by @id/name pair
    deduped = []
    seen = set()
    for item in candidates:
        key = (item.get("@id"), item.get("name"))
        if key not in seen:
            seen.add(key)
            deduped.append(item)
    return deduped

recordset_info = get_recordset_candidates(dataset, metadata_json)
recordset_df = pd.DataFrame(recordset_info)
recordset_df

### Record set discovery
If the table above is empty, it means the installed parser did not expose record-set definitions directly from the dataset metadata. In that case, the next cells provide a fallback workflow to inspect the dataset object and try candidate record set IDs interactively.

If record sets are present, we will use their `@id` values directly.

In [ ]:
# Inspect dataset object API to understand available methods and attributes
public_attrs = [attr for attr in dir(dataset) if not attr.startswith("_")]
public_attrs

## 3. Inspecting Records from a Record Set
The template notebook demonstrates iterating over `dataset.records(record_set=<id_of_the_record_set>)`.

Because record set IDs can vary by parser output, we first define a record set variable. If record sets were discovered automatically, we use the first available `@id`. Otherwise, you can replace the value manually after inspecting the metadata.

In [ ]:
# Select a record set @id dynamically when possible
record_set_ids = [item["@id"] for item in recordset_info if item.get("@id")]

selected_record_set_id = record_set_ids[0] if record_set_ids else None
print("Selected record set @id:", selected_record_set_id)

if selected_record_set_id is None:
    print(
        "No record set @id was discovered automatically. "
        "If needed, set `selected_record_set_id` manually after inspecting the Croissant schema."
    )

In [ ]:
# Preview a few records from the selected record set
preview_records = []

if selected_record_set_id is not None:
    try:
        for i, rec in enumerate(dataset.records(record_set=selected_record_set_id)):
            preview_records.append(rec)
            if i >= 4:
                break
    except Exception as e:
        print("Could not iterate records for the selected record set:", e)

preview_records

The preview above helps us understand the field names actually returned by `mlcroissant` for the selected record set. These field names commonly correspond to field `@id` values or resolved field labels, depending on the parser and dataset configuration.

## 4. Data Extraction
Now we load one or more record sets into pandas DataFrames.

To keep the notebook reusable, we iterate through all discovered record set `@id` values. If no record sets are available automatically, the notebook still shows the extraction logic and allows manual configuration.

In [ ]:
# Extract records from each discovered record set into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} rows from record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

print("\nLoaded record sets:", list(dataframes.keys()))

In [ ]:
# Display columns for the selected record set
if selected_record_set_id in dataframes:
    print("Columns in selected record set:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No DataFrame available for the selected record set yet.")

### DataFrame summary across record sets
The next cell provides a compact overview of the extracted tables.

In [ ]:
table_summary = []

for rs_id, df in dataframes.items():
    table_summary.append({
        "record_set_@id": rs_id,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(map(str, df.columns[:10])) + (" ..." if len(df.columns) > 10 else "")
    })

pd.DataFrame(table_summary).sort_values(by=["rows", "columns"], ascending=False)

## 5. Field Inspection and Type Inference
For exploratory analysis, it is useful to identify numeric, categorical, and boolean-like fields dynamically.

This is especially important for a general-purpose Croissant notebook because field availability depends on the record set selected.

In [ ]:
def infer_field_types(df):
    info = []
    for col in df.columns:
        series = df[col]
        coerced = pd.to_numeric(series, errors="coerce")
        numeric_ratio = coerced.notna().mean() if len(series) else 0
        inferred_type = "numeric" if numeric_ratio > 0.8 else str(series.dtype)
        info.append({
            "field_@id_or_name": col,
            "non_null": int(series.notna().sum()),
            "nulls": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True)),
            "inferred_type": inferred_type,
        })
    return pd.DataFrame(info)

if selected_record_set_id in dataframes:
    field_profile_df = infer_field_types(dataframes[selected_record_set_id])
    display(field_profile_df.sort_values(by=["inferred_type", "non_null"], ascending=[True, False]))
else:
    print("No selected DataFrame available for field inspection.")

## 6. Exploratory Data Analysis (EDA)
This section follows the template but makes the analysis dynamic so it can work with whichever record set is available.

We will:
1. detect a numeric field,
2. filter records above a threshold,
3. normalize the numeric field,
4. optionally group by a categorical field.

Where possible, fields are treated as field `@id` values returned by the loaded records.

In [ ]:
analysis_df = dataframes.get(selected_record_set_id, pd.DataFrame()).copy()

numeric_field = None
group_field = None

if not analysis_df.empty:
    # Identify numeric candidate fields
    numeric_candidates = []
    for col in analysis_df.columns:
        coerced = pd.to_numeric(analysis_df[col], errors="coerce")
        if coerced.notna().sum() > 0:
            numeric_candidates.append((col, coerced.notna().sum(), coerced.nunique(dropna=True)))

    # Prefer a field with enough numeric data and variation
    numeric_candidates = sorted(numeric_candidates, key=lambda x: (x[1], x[2]), reverse=True)
    if numeric_candidates:
        numeric_field = numeric_candidates[0][0]
        analysis_df[numeric_field] = pd.to_numeric(analysis_df[numeric_field], errors="coerce")

    # Identify a grouping field with a manageable number of categories
    categorical_candidates = []
    for col in analysis_df.columns:
        nunique = analysis_df[col].nunique(dropna=True)
        if 1 < nunique <= 20 and col != numeric_field:
            categorical_candidates.append((col, nunique))
    if categorical_candidates:
        group_field = sorted(categorical_candidates, key=lambda x: x[1])[0][0]

print("Selected numeric field:", numeric_field)
print("Selected group field:", group_field)

In [ ]:
if numeric_field is not None and not analysis_df.empty:
    valid_numeric = analysis_df[numeric_field].dropna()
    if not valid_numeric.empty:
        threshold = valid_numeric.quantile(0.5)
        filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        std = filtered_df[numeric_field].std()
        if pd.notna(std) and std != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std
            )
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0

        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field]
                .agg(["count", "mean", "median", "min", "max"])
                .sort_values(by="count", ascending=False)
            )
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head(10))
    else:
        print("No valid numeric values available for EDA.")
else:
    print("A suitable numeric field was not found for the selected record set.")

### Missing data analysis
Because the FAIR² dataset documentation explicitly discusses structured missingness, it is useful to inspect missing-value patterns in the selected table.

In [ ]:
if not analysis_df.empty:
    missing_df = pd.DataFrame({
        "field_@id_or_name": analysis_df.columns,
        "missing_count": analysis_df.isna().sum().values,
        "missing_pct": (analysis_df.isna().mean().values * 100).round(2)
    }).sort_values(by="missing_pct", ascending=False)
    display(missing_df.head(20))
else:
    print("No data available for missing-value inspection.")

## 7. Visualization
We now visualize the distribution of the selected numeric field and, when possible, compare it across categories from a grouping field.

In [ ]:
if numeric_field is not None and not analysis_df.empty:
    plot_df = analysis_df[[numeric_field]].copy()
    plot_df = plot_df.dropna()

    if not plot_df.empty:
        plt.figure(figsize=(10, 5))
        sns.histplot(plot_df[numeric_field], kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric data available for histogram plotting.")
else:
    print("No numeric field available for visualization.")

In [ ]:
if numeric_field is not None and group_field is not None and not analysis_df.empty:
    compare_df = analysis_df[[group_field, numeric_field]].copy().dropna()
    top_categories = compare_df[group_field].astype(str).value_counts().head(10).index
    compare_df = compare_df[compare_df[group_field].astype(str).isin(top_categories)]

    if not compare_df.empty:
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=compare_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough grouped data available for boxplot visualization.")
else:
    print("A compatible group field and numeric field were not both available for grouped visualization.")

## 8. Optional: Explore All Loaded Record Sets
If multiple record sets are available, the next cell provides a compact numeric summary for each one. This can help you decide which record set to analyze in more detail.

In [ ]:
all_profiles = []

for rs_id, df in dataframes.items():
    numeric_cols = 0
    for col in df.columns:
        if pd.to_numeric(df[col], errors="coerce").notna().sum() > 0:
            numeric_cols += 1
    all_profiles.append({
        "record_set_@id": rs_id,
        "rows": len(df),
        "columns": len(df.columns),
        "numeric_candidate_fields": numeric_cols,
    })

pd.DataFrame(all_profiles).sort_values(by=["rows", "numeric_candidate_fields"], ascending=False)

## 9. Conclusion
In this notebook, we:

- loaded the FAIR²-FoodHealth Croissant dataset with `mlcroissant`,
- inspected dataset-level metadata,
- discovered available record sets using their `@id` values when exposed,
- loaded records into pandas DataFrames,
- performed dynamic exploratory analysis on a selected record set,
- reviewed missing-value patterns, and
- visualized numeric distributions and grouped comparisons.

### Key observations
- This dataset is designed to be provenance-rich and governance-aware.
- Missingness may be meaningful and structured rather than accidental.
- Some records may represent synthetic, curated, source-derived, or guidance-oriented layers, so interpretation should respect dataset documentation and field semantics.

### Next steps
- Replace `selected_record_set_id` with a specific record set `@id` you want to study in detail.
- Identify field-level `@id` values for domain-specific analysis.
- Join related record sets for provenance-aware food-health workflows.
- Filter out synthetic or review-required records where appropriate for your use case.